# 第 4 章代码实验：闭包、装饰器与函数式工具

每组实验都按下面的顺序完成：

1. 先阅读代码，不要运行。
2. 把观察字典中的整个 `"__未填写__"`（包括引号）替换成预测的 Python 值。每个键都说明了它对应的表达式、变量或状态，但不会提示结果类型。
3. 运行当前实验，对照实际输出；如果预测有误，先解释原因再修改。
4. 立即运行紧随其后的“即时检查”。当前题通过后再进入下一题。
5. 用自己的话解释结果，并修改一个条件验证解释。

即时检查只核对可以客观判断的运行结果，不会代替原因解释。

遇到闭包时要标出局部名字和自由变量；遇到装饰器时要把装饰阶段与调用阶段分开记录。

## 实验 1：名字、函数对象与一等对象

预测函数对象绑定给别名、放入列表、作为实参和作为返回值时的结果。解释每个名字在相应时刻指向什么，哪些位置只读取函数对象，哪些位置使用括号发生调用；再指出谁是高阶函数，谁在调用中扮演回调。

In [4]:
def outer(function):
    print("decorate outer")

    def outer_wrapper():
        print("call outer")
        return "outer(" + function() + ")"

    return outer_wrapper


def inner(function):
    print("decorate inner")

    def inner_wrapper():
        print("call inner")
        return "inner(" + function() + ")"

    return inner_wrapper





@outer
@inner
def work():
    print("call work")
    return "done"


print("definition complete")
print(work())


decorate inner
decorate outer
definition complete
call outer
call inner
call work
outer(inner(done))


In [6]:
from functools import wraps


def make_trace(label):
    print("factory", label)

    def decorator(function):
        print("decorate", label)

        @wraps(function)
        def wrapper(*args, **kwargs):
            print("enter", label)
            result = function(*args, **kwargs)
            print("leave", label)
            return result

        return wrapper

    return decorator


@make_trace("outer")
@make_trace("inner")
def work():
    print("work")

print("definition complete")
work()


factory outer
factory inner
decorate inner
decorate outer
definition complete
enter outer
enter inner
work
leave inner
leave outer


In [2]:
def inner(function):
    def wrapper():
        print('inner')
        return "inner(" + function() + ")"

    return wrapper


def outer(function):
    def wrapper():
        print('outer')
        return "outer(" + function() + ")"

    return wrapper


@outer
@inner
def work():
    print('work')
    return "done"


print(work())  # outer(inner(done))


outer
inner
work
outer(inner(done))


In [5]:
from functools import wraps


def repeat(times):
    def decorator(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = function(*args, **kwargs)
            return result

        return wrapper

    return decorator


def greet(name):
    return f"Hello, {name}"


decorator = repeat(3)
greet = decorator(greet)
result = greet("Ming")
print(result)


Hello, Ming


In [4]:
from functools import wraps


def repeat(times):
    def decorator(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = function(*args, **kwargs)
            return result

        return wrapper

    return decorator


@repeat(3)
def greet(name):
    return f"Hello, {name}"


print(greet("Ming"))

Hello, Ming


In [3]:
def trace(function):
    print("decorating")

    def wrapper(*args, **kwargs):
        print("before")
        result = function(*args, **kwargs)
        print("after")
        return result

    return wrapper


@trace
def greet(name):
    return f"Hello, {name}"


print("defined")
print(greet("Ming"))


decorating
defined
before
after
Hello, Ming


In [1]:
def trace(function):
    def wrapper(*args, **kwargs):
        print("before")
        result = function(*args, **kwargs)
        print("after")
        return result

    return wrapper


def greet(name):
    return f"Hello, {name}"


greet = trace(greet)
greet = trace(greet)
print(greet("Ming"))


before
before
after
after
Hello, Ming


In [8]:
def greet(name):
    return f"Hello, {name}"


def traced_greet(name):
    print("before")
    result = greet(name)
    print("after")
    return result
traced_greet('hhh')

before
after


'Hello, hhh'

In [ ]:
def make_late_readers():
    functions = []
    for number in range(3):
        def read():
            return number

        functions.append(read)
    return functions


functions = make_late_readers()
print(functions[0] is functions[1])  # False
for function in functions:
    print(function())


False
False
2
2
2


In [5]:
def make_fixed_readers():
    functions = []
    for number in range(3):
        def read(number=number):
            return number

        functions.append(read)
    return functions


functions = make_fixed_readers()
for function in functions:
    print(function())


0
1
2


In [1]:
def make_multiplier(factor):
    def multiply(value):
        return value * factor

    return multiply


double = make_multiplier(2)
triple = make_multiplier(3)

print(double(10))
print(triple(10))


20
30


In [2]:
double = make_multiplier(2)

print(double.__code__.co_freevars)
print(double.__closure__)


('factor',)
(<cell at 0x0000028601FF5360: int object at 0x00007FFB7BA5D9D8>,)


In [ ]:
def increase(number):
    return number + 1

def double(number):
    return number * 2

def apply(operation, value):
    return operation(value)

def choose_operation(use_double):
    if use_double:
        return double
    return increase

alias = increase
operations = [increase, double]
stored_results = [operation(3) for operation in operations]
higher_order_result = apply(double, 4)
chosen = choose_operation(True)
returned_function_is_double = chosen is double
print(alias is increase, stored_results, higher_order_result)
print(returned_function_is_double, chosen(5))

observation_01 = {
    "01. alias is increase 的结果": "__未填写__",
    "02. 依次调用 operations 中函数的结果": "__未填写__",
    "03. apply(double, 4) 的结果": "__未填写__",
    "04. chosen is double 的结果": "__未填写__",
    "05. chosen(5) 的结果": "__未填写__",
}

### 实验 1 即时检查

先运行上面的实验单元格，再运行下面的检查。它只检查当前实验；其他实验尚未填写不会影响本次结果。


In [ ]:
from chapter4_checks import check_experiment

check_experiment(globals(), 1)


### 实验 1 原因解释

请用自己的话区分名字、函数对象和调用结果；说明四种操作分别在哪里发生，并指出 `apply` 与传入的 `double` 在本次调用中各自扮演什么角色。

## 实验 2：返回内层函数形成闭包

画出 `double`、`triple` 各自需要的 `factor` 绑定。说明为什么外层调用结束后仍能得到结果。

In [ ]:
def make_multiplier(factor):
    def multiply(value):
        return value * factor

    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)
double_result = double(10)
triple_result = triple(10)
print(double_result, triple_result, double is not triple)

observation_02 = {
    "01. double(10) 的结果": "__未填写__",
    "02. triple(10) 的结果": "__未填写__",
    "03. double is not triple 的结果": "__未填写__",
}

### 实验 2 即时检查

先运行上面的实验单元格，再运行下面的检查。它只检查当前实验；其他实验尚未填写不会影响本次结果。


In [ ]:
from chapter4_checks import check_experiment

check_experiment(globals(), 2)


### 实验 2 原因解释

请分别画出两次 `make_multiplier()` 调用中的 `factor` 绑定，并说明调用外层函数、返回内层函数和以后调用内层函数发生在什么时刻。

## 实验 3：同次调用共享状态，不同调用互相独立

解释为什么同一个 `make_box()` 返回的 `add` 和 `read` 共享列表，而两次 `make_box()` 的列表互不影响。

In [ ]:
def make_box():
    values = []

    def add(value):
        values.append(value)

    def read():
        return values.copy()

    return add, read

first_add, first_read = make_box()
second_add, second_read = make_box()
first_add("A")
first_add("B")
second_add("X")
first_box = first_read()
second_box = second_read()
print(first_box, second_box, first_read())

observation_03 = {
    "01. first_read() 读取到的列表": "__未填写__",
    "02. second_read() 读取到的列表": "__未填写__",
    "03. 再次调用 first_read() 的结果": "__未填写__",
}

### 实验 3 即时检查

先运行上面的实验单元格，再运行下面的检查。它只检查当前实验；其他实验尚未填写不会影响本次结果。


In [ ]:
from chapter4_checks import check_experiment

check_experiment(globals(), 3)


### 实验 3 原因解释

请说明同一次 `make_box()` 返回的两个函数为什么共享状态，而两次 `make_box()` 调用建立的状态为什么互不影响。

## 实验 4：`nonlocal` 重新绑定与可变对象修改

指出为什么计数器需要 `nonlocal`，而列表收集器不需要。

In [ ]:
def make_counter():
    count = 0

    def increase_count():
        nonlocal count
        count += 1
        return count

    return increase_count

def make_collector():
    values = []

    def collect(value):
        values.append(value)
        return values.copy()

    return collect

counter = make_counter()
collector = make_collector()
counter_results = (counter(), counter())
collector_results = (collector("A"), collector("B"))
print(counter_results, collector_results)

observation_04 = {
    "01. 连续两次调用 counter() 的结果": "__未填写__",
    "02. 连续两次调用 collector() 的结果": "__未填写__",
}

### 实验 4 即时检查

先运行上面的实验单元格，再运行下面的检查。它只检查当前实验；其他实验尚未填写不会影响本次结果。


In [ ]:
from chapter4_checks import check_experiment

check_experiment(globals(), 4)


### 实验 4 原因解释

请逐行判断代码是在重新绑定外层名字，还是修改外层名字指向的列表对象，并据此解释哪一处需要 `nonlocal`。

## 实验 5：观察自由变量和闭包单元格

这些特殊属性只用于观察，不要求记忆内部结构。预测捕获名称、单元格数量和内容。

In [ ]:
reader = make_multiplier(4)
free_variable_names = reader.__code__.co_freevars
closure_cell_count = len(reader.__closure__)
captured_value = reader.__closure__[0].cell_contents

def plain_function(value):
    return value * 2

plain_function_closure = plain_function.__closure__
print(free_variable_names, closure_cell_count, captured_value)
print(plain_function_closure)

observation_05 = {
    "01. reader 捕获的自由变量名称": "__未填写__",
    "02. reader 的闭包单元格数量": "__未填写__",
    "03. 闭包中捕获的 factor 值": "__未填写__",
    "04. plain_function.__closure__ 的值": "__未填写__",
}

### 实验 5 即时检查

先运行上面的实验单元格，再运行下面的检查。它只检查当前实验；其他实验尚未填写不会影响本次结果。


In [ ]:
from chapter4_checks import check_experiment

check_experiment(globals(), 5)


### 实验 5 原因解释

请说明自由变量名称、闭包单元格和其中对象之间的关系，以及为什么这些属性适合观察而不适合成为业务接口。

## 实验 6：循环闭包的晚绑定

不要先运行。画出三个函数是否拥有三个独立循环值，还是共享同一个外层绑定。

In [ ]:
def make_late_readers():
    readers = []
    for number in range(3):
        readers.append(lambda: number)
    return readers

late_readers = make_late_readers()
late_bound_results = [reader() for reader in late_readers]
print(late_bound_results)

observation_06 = {
    "01. 三个晚绑定函数的调用结果": "__未填写__",
}

### 实验 6 即时检查

先运行上面的实验单元格，再运行下面的检查。它只检查当前实验；其他实验尚未填写不会影响本次结果。


In [ ]:
from chapter4_checks import check_experiment

check_experiment(globals(), 6)


### 实验 6 原因解释

请画出三个函数读取的 `number` 是否来自同一绑定，并区分函数创建时刻、循环结束时刻和函数调用时刻。

## 实验 7：两种晚绑定修复

分别解释形参默认值和函数工厂怎样改变每个函数读取值的来源。

In [ ]:
def make_default_fixed_readers():
    readers = []
    for number in range(3):
        readers.append(lambda number=number: number)
    return readers

def make_reader(value):
    def read():
        return value
    return read

default_fixed = [reader() for reader in make_default_fixed_readers()]
factory_fixed = [make_reader(number)() for number in range(3)]
print(default_fixed, factory_fixed)

observation_07 = {
    "01. 使用形参默认值修复后的三个结果": "__未填写__",
    "02. 使用函数工厂修复后的三个结果": "__未填写__",
}

### 实验 7 即时检查

先运行上面的实验单元格，再运行下面的检查。它只检查当前实验；其他实验尚未填写不会影响本次结果。


In [ ]:
from chapter4_checks import check_experiment

check_experiment(globals(), 7)


### 实验 7 原因解释

请分别说明形参默认值和函数工厂如何改变每个函数读取数据的来源，而不是笼统地说它们关闭了晚绑定。

## 实验 8：装饰阶段与调用阶段

先写出执行函数定义后已有的事件，再写出调用后的完整事件。

In [ ]:
events = []

def mark(function):
    events.append("decorate")

    def wrapper(*args, **kwargs):
        events.append("before")
        result = function(*args, **kwargs)
        events.append("after")
        return result

    return wrapper

@mark
def greet(name):
    events.append("function")
    return f"Hello, {name}"

events_after_definition = events.copy()
call_result = greet("Ming")
events_after_call = events.copy()
print(events_after_definition, call_result, events_after_call)

observation_08 = {
    "01. greet 定义完成后的 events": "__未填写__",
    "02. greet('Ming') 的返回值": "__未填写__",
    "03. greet 调用完成后的 events": "__未填写__",
}

### 实验 8 即时检查

先运行上面的实验单元格，再运行下面的检查。它只检查当前实验；其他实验尚未填写不会影响本次结果。


In [ ]:
from chapter4_checks import check_experiment

check_experiment(globals(), 8)


### 实验 8 原因解释

请把函数定义、装饰器应用和实际调用分成三个阶段，说明每个阶段执行了哪个函数体以及名字 `greet` 最终指向什么。

## 实验 9：转发实参和返回结果

观察包装器收到的位置实参、关键字实参和最终返回值。

In [ ]:
seen_calls = []

def capture_call(function):
    def wrapper(*args, **kwargs):
        seen_calls.append((args, kwargs.copy()))
        return function(*args, **kwargs)
    return wrapper

@capture_call
def add(left, right=0):
    return left + right

captured_result = add(2, right=3)
seen_args, seen_kwargs = seen_calls[0]
print(captured_result, seen_args, seen_kwargs)

observation_09 = {
    "01. add(2, right=3) 的返回值": "__未填写__",
    "02. 包装器收到的位置实参": "__未填写__",
    "03. 包装器收到的关键字实参": "__未填写__",
}

### 实验 9 即时检查

先运行上面的实验单元格，再运行下面的检查。它只检查当前实验；其他实验尚未填写不会影响本次结果。


In [ ]:
from chapter4_checks import check_experiment

check_experiment(globals(), 9)


### 实验 9 原因解释

请说明包装器怎样收集位置实参和关键字实参、怎样展开给原函数，以及为什么还必须返回原函数结果。

## 实验 10：`wraps` 与函数元数据

比较没有和使用 `wraps` 的两个函数名称、文档及 `__wrapped__`。

In [ ]:
from functools import wraps

def plain_decorator(function):
    def wrapper(*args, **kwargs):
        return function(*args, **kwargs)
    return wrapper

def preserving_decorator(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        return function(*args, **kwargs)
    return wrapper

@plain_decorator
def plain_target():
    """这段文档会被遮蔽。"""
    return 1

@preserving_decorator
def documented():
    """保留这段文档。"""
    return 2

print(plain_target.__name__, plain_target.__doc__)
print(documented.__name__, documented.__doc__, hasattr(documented, "__wrapped__"))

observation_10 = {
    "01. 未使用 wraps 时的函数名": "__未填写__",
    "02. 未使用 wraps 时的函数文档": "__未填写__",
    "03. 使用 wraps 后的函数名": "__未填写__",
    "04. 使用 wraps 后的函数文档": "__未填写__",
    "05. 是否保留 __wrapped__ 引用": "__未填写__",
}

### 实验 10 即时检查

先运行上面的实验单元格，再运行下面的检查。它只检查当前实验；其他实验尚未填写不会影响本次结果。


In [ ]:
from chapter4_checks import check_experiment

check_experiment(globals(), 10)


### 实验 10 原因解释

请比较两个装饰后名字实际指向的对象，并说明 `wraps` 保留了哪些观察信息、又没有自动修复哪些调用逻辑。

## 实验 11：带配置的装饰器

标出配置、装饰和调用三层。预测原函数实际执行次数和包装器返回结果。

In [ ]:
repeat_calls = []

def repeat(times):
    def decorator(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = function(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def repeated_greet(name):
    repeat_calls.append(name)
    return f"Hello, {name}"

repeat_result = repeated_greet("Ming")
print(repeat_result, len(repeat_calls))

observation_11 = {
    "01. repeated_greet('Ming') 的返回值": "__未填写__",
    "02. 原函数实际执行次数": "__未填写__",
}

### 实验 11 即时检查

先运行上面的实验单元格，再运行下面的检查。它只检查当前实验；其他实验尚未填写不会影响本次结果。


In [ ]:
from chapter4_checks import check_experiment

check_experiment(globals(), 11)


### 实验 11 原因解释

请标出配置、装饰和调用三层各自接收什么、返回什么、何时执行，并说明配置值由哪一层闭包保存。

## 实验 12：多个装饰器的顺序

先记录装饰阶段，再清空事件并记录调用阶段。不要用一句“从下到上”同时回答两个阶段。

In [ ]:
layer_events = []

def make_layer(label):
    def decorator(function):
        layer_events.append(("decorate", label))

        @wraps(function)
        def wrapper(*args, **kwargs):
            layer_events.append(("enter", label))
            result = function(*args, **kwargs)
            layer_events.append(("leave", label))
            return result
        return wrapper
    return decorator

@make_layer("outer")
@make_layer("inner")
def layered_work():
    layer_events.append(("work", None))
    return "done"

definition_events = layer_events.copy()
layer_events.clear()
layered_work()
call_events = layer_events.copy()
print(definition_events)
print(call_events)

observation_12 = {
    "01. 两个装饰器的装饰阶段事件": "__未填写__",
    "02. 两个装饰器的调用阶段事件": "__未填写__",
}

### 实验 12 即时检查

先运行上面的实验单元格，再运行下面的检查。它只检查当前实验；其他实验尚未填写不会影响本次结果。


In [ ]:
from chapter4_checks import check_experiment

check_experiment(globals(), 12)


### 实验 12 原因解释

请分别写出装饰器表达式求值、装饰器应用和包装器调用的顺序，解释为什么一句“从下到上”不能覆盖全部阶段。

## 实验 13：类装饰器接收与返回类对象

当前只观察类对象被传给 `register`。完整类和实例机制在第 6 章学习。

In [ ]:
class_registry = []

def register(class_object):
    class_registry.append(class_object.__name__)
    return class_object

@register
class Report:
    pass

report = Report()
print(class_registry, Report.__name__, isinstance(report, Report))

observation_13 = {
    "01. register 记录的类名": "__未填写__",
    "02. Report.__name__ 的结果": "__未填写__",
    "03. isinstance(report, Report) 的结果": "__未填写__",
}

### 实验 13 即时检查

先运行上面的实验单元格，再运行下面的检查。它只检查当前实验；其他实验尚未填写不会影响本次结果。


In [ ]:
from chapter4_checks import check_experiment

check_experiment(globals(), 13)


### 实验 13 原因解释

请说明执行类定义后得到的对象如何传给 `register`，以及为什么装饰器原样返回该对象后 `Report()` 仍可创建实例。

## 实验 14：`lambda`、`map()`、`filter()` 与推导式

预测所有结果。说明 `list(mapped)` 和 `list(filtered)` 为什么会真正取得全部按需结果。完整迭代机制在第 5 章学习。

In [ ]:
square = lambda number: number * number
numbers = [1, 2, 3, 4]
mapped_values = list(map(lambda number: number * 2, numbers))
filtered_values = list(filter(lambda number: number % 2 == 0, numbers))
comprehension_values = [number * 2 for number in numbers if number % 2 == 0]
print(square(5), mapped_values, filtered_values, comprehension_values)

observation_14 = {
    "01. square(5) 的结果": "__未填写__",
    "02. map 转成列表后的结果": "__未填写__",
    "03. filter 转成列表后的结果": "__未填写__",
    "04. 列表推导式的结果": "__未填写__",
}

### 实验 14 即时检查

先运行上面的实验单元格，再运行下面的检查。它只检查当前实验；其他实验尚未填写不会影响本次结果。


In [ ]:
from chapter4_checks import check_experiment

check_experiment(globals(), 14)


### 实验 14 原因解释

请说明 `lambda` 在何时创建函数对象，以及 `map()`、`filter()` 分别怎样调用传入函数；再比较它们与推导式表达的过程。

## 实验 15：`reduce()` 与 `partial()`

逐步写出累计值变化，并说明 `partial()` 返回新可调用对象时是否已经执行原函数。

In [ ]:
from functools import partial, reduce

reduced_total = reduce(lambda total, value: total + value, [1, 2, 3, 4], 0)
empty_total = reduce(lambda total, value: total + value, [], 0)

def power(base, exponent):
    return base**exponent

square_partial = partial(power, exponent=2)
partial_result = square_partial(5)
print(reduced_total, empty_total, partial_result)

observation_15 = {
    "01. reduce 对 [1, 2, 3, 4] 的结果": "__未填写__",
    "02. reduce 对空列表的结果": "__未填写__",
    "03. square_partial(5) 的结果": "__未填写__",
}

### 实验 15 即时检查

先运行上面的实验单元格，再运行下面的检查。它只检查当前实验；其他实验尚未填写不会影响本次结果。


In [ ]:
from chapter4_checks import check_experiment

check_experiment(globals(), 15)


### 实验 15 原因解释

请写出 `reduce()` 的累计值变化，并说明创建 `partial` 对象与以后调用它分别发生了什么。

## 实验 16：`lru_cache()` 命中、可哈希实参与可变结果

预测真实函数体执行次数、命中统计和对象身份。再说明为什么修改缓存返回的字典会影响后续相同调用。

In [ ]:
from functools import lru_cache

real_calls = []

@lru_cache(maxsize=2)
def build_result(number):
    real_calls.append(number)
    return {"value": number * number}

first_result = build_result(2)
initial_value = first_result["value"]
second_result = build_result(2)
same_cached_object = first_result is second_result
cache_information = build_result.cache_info()
first_result["changed"] = True
cached_after_mutation = build_result(2).copy()

@lru_cache(maxsize=2)
def cached_total(values):
    return sum(values)

unhashable_error = None
try:
    cached_total([1, 2])
except Exception as error:
    unhashable_error = type(error).__name__

print(initial_value, same_cached_object, real_calls)
print(cache_information.hits, cache_information.misses)
print(cached_after_mutation, unhashable_error)

observation_16 = {
    "01. 第一次 build_result(2) 得到的 value": "__未填写__",
    "02. 两次调用是否返回同一个缓存对象": "__未填写__",
    "03. 真实函数体收到过的实参": "__未填写__",
    "04. 缓存命中次数": "__未填写__",
    "05. 缓存未命中次数": "__未填写__",
    "06. 修改首次结果后再次取得的缓存内容": "__未填写__",
    "07. 列表作为缓存实参时的异常类型": "__未填写__",
}

### 实验 16 即时检查

先运行上面的实验单元格，再运行下面的检查。它只检查当前实验；其他实验尚未填写不会影响本次结果。


In [ ]:
from chapter4_checks import check_experiment

check_experiment(globals(), 16)


### 实验 16 原因解释

请区分缓存未命中与命中，说明为什么相同实参会得到同一个可变结果对象，以及列表实参为什么不能参与缓存键。

## 全部复查

完成本章全部实验后再运行下面的单元格。这里用于整章复查，不能代替每道实验后面的即时检查。


In [ ]:
from chapter4_checks import run_checks

run_checks(globals())
